In [1]:
!pip install -U evaluate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.4 MB/s eta 0:00:00


In [2]:
!pip install -q bitsandbytes peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 41.7 MB/s eta 0:00:00:00:0100:01


In [27]:
import json
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    TrainerCallback
)
import logging
from transformers.utils import logging as hf_logging

import numpy as np
import evaluate
import torch
import os

# Paths: adjust to your dataset mount
DATA_DIR = "/kaggle/input/kharagpur-data-science-hackathon-kdsh-2026-dataset" 
# TRAIN_PATH = f"{DATA_DIR}/train.csv"
TRAIN_PATH="/kaggle/input/kdsh26-train-augmented-dataset/train_augmented_50k_balanced.csv"
TEST_PATH  = f"{DATA_DIR}/test.csv"

MC_CONS_PATH  = "/kaggle/input/kdsh26-jsonl-file-characters/monte_cristo/monte_cristo_constraints_updated.jsonl"
CA_CONS_PATH  = "/kaggle/input/kdsh26-jsonl-file-characters/castaways/castaways_constraints_filled.jsonl"

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
print(train_df.head())
print(train_df["label"].value_counts())


   id                   book_name                   char  \
0   1  In Search of the Castaways           John Mangles   
1   2  In Search of the Castaways          Captain Grant   
2   3  In Search of the Castaways           John Mangles   
3   4  In Search of the Castaways        Jacques Paganel   
4   5  In Search of the Castaways  Lady Helena Glenarvan   

                caption                                            content  \
0  geographic_expertise  In In Search of the Castaways, John Mangles's ...   
1                  role  In In Search of the Castaways, Captain Grant i...   
2  geographic_expertise  In In Search of the Castaways, it is NOT true ...   
3                   NaN  At twelve, Jacques Paganel fell in love with g...   
4                  role  In In Search of the Castaways, Lady Helena Gle...   

        label  
0  consistent  
1  contradict  
2  contradict  
3  consistent  
4  consistent  
label
consistent    26428
contradict    23572
Name: count, dtype: int64


In [23]:
logging.basicConfig(level=logging.INFO)
hf_logging.set_verbosity_info()
hf_logging.enable_default_handler()
hf_logging.enable_explicit_format()

In [24]:
gradient_checkpointing=False
os.environ["HF_DISABLE_PROGRESS_BARS"] = "0"
os.environ["DISABLE_TQDM"] = "0"

In [4]:
def load_constraints(path):
    mapping = {}
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            key = (obj["book_name"], obj["character"])
            mapping[key] = obj.get("constraints", [])
    return mapping

mc_constraints = load_constraints(MC_CONS_PATH)  # Monte Cristo [code_file:262]
ca_constraints = load_constraints(CA_CONS_PATH)  # Castaways    [code_file:293]
constraints = {**mc_constraints, **ca_constraints}

def constraint_to_sentence(book, char, c):
    dim = c["dimension"]
    val = c["value"]
    if dim == "health_state":
        return f"In {book}, {char} is {val}."
    elif dim == "family_role":
        return f"In {book}, {char} has family role: {val}."
    elif dim == "role":
        return f"In {book}, {char} is described as {val}."
    elif dim == "geographic_expertise":
        return f"{char} is familiar with {val}."
    elif dim == "criminal_history":
        return f"{char} has criminal history: {val}."
    else:
        return f"{dim}: {val}."

def build_context(book, char, max_cons=6):
    cons = constraints.get((book, char), [])
    if not cons:
        return ""
    sents = [constraint_to_sentence(book, char, c) for c in cons[:max_cons]]
    return " ".join(sents)

train_df["context"] = train_df.apply(
    lambda r: build_context(r["book_name"], r["char"]), axis=1
)
test_df["context"] = test_df.apply(
    lambda r: build_context(r["book_name"], r["char"]), axis=1
)

# Quick check
print(train_df[["book_name","char","context","content","label"]].head())


                    book_name                   char  \
0  In Search of the Castaways           John Mangles   
1  In Search of the Castaways          Captain Grant   
2  In Search of the Castaways           John Mangles   
3  In Search of the Castaways        Jacques Paganel   
4  In Search of the Castaways  Lady Helena Glenarvan   

                                             context  \
0  In In Search of the Castaways, John Mangles is...   
1  In In Search of the Castaways, Captain Grant i...   
2  In In Search of the Castaways, John Mangles is...   
3  In In Search of the Castaways, Jacques Paganel...   
4  In In Search of the Castaways, Lady Helena Gle...   

                                             content       label  
0  In In Search of the Castaways, John Mangles's ...  consistent  
1  In In Search of the Castaways, Captain Grant i...  contradict  
2  In In Search of the Castaways, it is NOT true ...  contradict  
3  At twelve, Jacques Paganel fell in love with g...  cons

In [5]:
from transformers import BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"  # or "Qwen/Qwen2.5-7B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
# Qwen often has no pad_token by default; align it:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4‑bit quantization config (QLoRA style)[web:510][web:513]
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Load base in 4‑bit, with 2‑class head
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    quantization_config=bnb_config,
    device_map="auto",
    ignore_mismatched_sizes=True,  # needed since base is a causal LM
)

label2id = {"consistent": 0, "contradict": 1}
id2label = {v: k for k, v in label2id.items()}
model.config.label2id = label2id
model.config.id2label = id2label

# Attach LoRA adapter for sequence classification[web:513]
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,  # important: sequence classification task
)

model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable()
model.print_trainable_parameters()  # sanity check: only a few M params trainable



def encode_examples(df, is_train=True):
    texts = [
        f"Premise: {ctx}\nHypothesis: {claim}"
        for ctx, claim in zip(df["context"].fillna(""), df["content"])
    ]
    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=512,
    )
    if is_train:
        enc["labels"] = [label2id[l] for l in df["label"]]
    return enc

# Train/val split (stratified)
train_split, val_split = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["label"],
    random_state=42,
)

train_enc = encode_examples(train_split, is_train=True)
val_enc   = encode_examples(val_split,   is_train=True)
test_enc  = encode_examples(test_df,    is_train=False)

train_ds = Dataset.from_dict(train_enc)
val_ds   = Dataset.from_dict(val_enc)
test_ds  = Dataset.from_dict(test_enc)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

...

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen2.5-7B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 5,053,440 || all params: 7,075,679,744 || trainable%: 0.0714


In [6]:
import numpy as np
import torch

# Compute class weights from the training split
train_label_ids = train_split["label"].map(label2id).values  # label2id = {"consistent": 0, "contradict": 1}

class_counts = np.bincount(train_label_ids)  # e.g. [#consistent, #contradict]
num_classes = len(class_counts)

# Inverse-frequency weights, normalized so mean ≈ 1
weights = class_counts.sum() / (num_classes * class_counts.astype(np.float32))
class_weights = torch.tensor(
    weights, dtype=torch.float32
).to("cuda" if torch.cuda.is_available() else "cpu")

print("Class counts:", class_counts)
print("Class weights:", class_weights)


Class counts: [21142 18858]
Class weights: tensor([0.9460, 1.0606], device='cuda:0')


In [14]:
training_args = TrainingArguments(
    output_dir="./qwen2.5-7b-books-lora-cls",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    bf16=True,
    gradient_checkpointing=True,

    # logging / tqdm
    logging_strategy="steps",   # <- important
    logging_steps=1,            # log every optimizer step
    logging_first_step=True,    # show the very first step
    disable_tqdm=False,         # enable progress bar
    report_to="none",
)


In [15]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred  # [batch, 2]
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
    }

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits  # [batch, 2]
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


In [28]:
class StepLoggingCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % 10 == 0:
            print(f"[Step {state.global_step}] Loss: {state.log_history[-1].get('loss', 'N/A'):.4f}")

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[StepLoggingCallback()],  # Add this
)

/tmp/ipykernel_106/704488171.py:6: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  trainer = WeightedTrainer(
[INFO|trainer.py:749] 2026-01-10 12:56:46,610 >> Using auto half precision backend


In [ ]:
trainer.train()

SAVE_DIR = "./kaggle/working/qwen2.5-7b-books-lora-cls"
trainer.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Saved fine-tuned model + tokenizer to", SAVE_DIR)


[INFO|trainer.py:2519] 2026-01-10 12:56:57,580 >> ***** Running training *****
[INFO|trainer.py:2520] 2026-01-10 12:56:57,581 >>   Num examples = 40,000
[INFO|trainer.py:2521] 2026-01-10 12:56:57,581 >>   Num Epochs = 3
[INFO|trainer.py:2522] 2026-01-10 12:56:57,582 >>   Instantaneous batch size per device = 1
[INFO|trainer.py:2525] 2026-01-10 12:56:57,582 >>   Total train batch size (w. parallel, distributed & accumulation) = 8
[INFO|trainer.py:2526] 2026-01-10 12:56:57,582 >>   Gradient Accumulation steps = 8
[INFO|trainer.py:2527] 2026-01-10 12:56:57,582 >>   Total optimization steps = 15,000
[INFO|trainer.py:2528] 2026-01-10 12:56:57,584 >>   Number of trainable parameters = 5,053,440
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.401600,0.230340,0.916500,0.915578


[Step 10] Loss: 10.1732
[Step 20] Loss: 6.3420
[Step 30] Loss: 3.9219
[Step 40] Loss: 7.4275
[Step 50] Loss: 4.0290
[Step 60] Loss: 7.1122
[Step 70] Loss: 4.2463
[Step 80] Loss: 5.2368
[Step 90] Loss: 4.9461
[Step 100] Loss: 7.8340
[Step 110] Loss: 6.4742
[Step 120] Loss: 7.8234
[Step 130] Loss: 8.7780
[Step 140] Loss: 9.8210
[Step 150] Loss: 6.2716
[Step 160] Loss: 3.4685
[Step 170] Loss: 6.9531
[Step 180] Loss: 7.9904
[Step 190] Loss: 5.8748
[Step 200] Loss: 2.3136
[Step 210] Loss: 5.4134
[Step 220] Loss: 7.7578
[Step 230] Loss: 6.8452
[Step 240] Loss: 7.2351
[Step 250] Loss: 13.8034
[Step 260] Loss: 12.5494
[Step 270] Loss: 16.7678
[Step 280] Loss: 6.9094
[Step 290] Loss: 4.5945
[Step 300] Loss: 9.6328
[Step 310] Loss: 3.6602
[Step 320] Loss: 8.8184
[Step 330] Loss: 6.5865
[Step 340] Loss: 5.0298
[Step 350] Loss: 10.6874
[Step 360] Loss: 2.5594
[Step 370] Loss: 3.7050
[Step 380] Loss: 2.5232
[Step 390] Loss: 6.3285
[Step 400] Loss: 6.2419
[Step 410] Loss: 9.6086
[Step 420] Loss: 3.5

[INFO|trainer.py:4643] 2026-01-10 13:44:36,744 >> 
***** Running Evaluation *****
[INFO|trainer.py:4645] 2026-01-10 13:44:36,745 >>   Num examples = 10000
[INFO|trainer.py:4648] 2026-01-10 13:44:36,745 >>   Batch size = 2
[INFO|trainer.py:4309] 2026-01-10 13:49:57,332 >> Saving model checkpoint to ./qwen2.5-7b-books-lora-cls/checkpoint-5000
[INFO|configuration_utils.py:765] 2026-01-10 13:49:57,685 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28/config.json
[INFO|configuration_utils.py:839] 2026-01-10 13:49:57,686 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 3584,
  "initializer_range": 0.02,
  "intermediate_size": 18944,
  "layer_types": [
    "full_attention",
    "full_attention",
    "ful

[Step 5010] Loss: 3.8624
[Step 5020] Loss: 0.8324
[Step 5030] Loss: 3.4468
[Step 5040] Loss: 4.4526
[Step 5050] Loss: 0.2713
[Step 5060] Loss: 0.5692
[Step 5070] Loss: 1.1883
[Step 5080] Loss: 3.9438
[Step 5090] Loss: 0.1192
[Step 5100] Loss: 2.1431
[Step 5110] Loss: 0.6676
[Step 5120] Loss: 2.5145
[Step 5130] Loss: 0.2553
[Step 5140] Loss: 0.4296
[Step 5150] Loss: 2.3927
[Step 5160] Loss: 0.5242
[Step 5170] Loss: 1.8681
[Step 5180] Loss: 1.3210
[Step 5190] Loss: 1.0045
[Step 5200] Loss: 2.4211
[Step 5210] Loss: 0.9274
[Step 5220] Loss: 0.1653
[Step 5230] Loss: 3.7696
[Step 5240] Loss: 1.5843
[Step 5250] Loss: 5.6196
[Step 5260] Loss: 0.2774
[Step 5270] Loss: 0.5099
[Step 5280] Loss: 2.5524
[Step 5290] Loss: 1.3261
[Step 5300] Loss: 2.6975
[Step 5310] Loss: 0.2055
[Step 5320] Loss: 0.7323
[Step 5330] Loss: 2.8655
[Step 5340] Loss: 0.4190
[Step 5350] Loss: 2.8165
[Step 5360] Loss: 0.1429
[Step 5370] Loss: 2.4887
[Step 5380] Loss: 7.7952
[Step 5390] Loss: 1.2558
[Step 5400] Loss: 3.7225


In [10]:
label_map_path = os.path.join(SAVE_DIR, "label_map.json")
with open(label_map_path, "w") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f)

NameError: name 'os' is not defined